# DEMETER STSM — Rolling-Window Training with LSTM / GRU, Bidirectional and Attention Encoders

Same conditions as `DEMETER_LSTM_AE-Training.ipynb`; only the autoencoder changes, and here
the run uses 20 windows and a cosine-annealed learning rate.

A model is `(CELL, BIDIRECTIONAL, ATTENTION)`:

| `CELL` | `BIDIRECTIONAL` | `ATTENTION` | Name |
|---|---|---|---|
| `"LSTM"` / `"GRU"` | `False` | `None` | `LSTM`, `GRU` |
| `"LSTM"` / `"GRU"` | `True` | `None` | `BiLSTM`, `BiGRU` |
| `"LSTM"` / `"GRU"` | `False` / `True` | `"temporal"` | `LSTM-TA`, `GRU-TA`, `BiLSTM-TA`, `BiGRU-TA` |
| `"LSTM"` / `"GRU"` | `False` / `True` | `"feature_temporal"` | `LSTM-FTA`, `GRU-FTA`, `BiLSTM-FTA`, `BiGRU-FTA` |

- **TA** (temporal attention): the encoder summary is a softmax-weighted sum of all encoder outputs
  over time, instead of the last hidden state.
- **FTA** (feature + temporal attention): softmax weights over the 11 input features at each time step, then TA.

Every model in `MODELS` is trained in turn, each starting from the same seed. Each one writes to
`Clean-Code/STSM-Results/<name>/<run label>/`, and every saved file carries its run label
(e.g. `tw48-30dBG_A-BiGRU-FTA`).

For each rolling window (12 months train, 3 months validation, 3-month stride):

1. Load the background-corrected window data and split it into train / validation.
2. Build half-orbit pair datasets: **all pairs** and **normal pairs** (seismic label 0).
3. Fit one scaler on the normal training pairs and apply it to all four datasets.
4. Train the autoencoder on normal train pairs, with early stopping on normal validation pairs.
   With `WARM_START`, window *i > 0* starts from the model saved for window *i − 1*.
5. Evaluate on **all** train and validation pairs: aggregate reconstruction error, threshold at the
   `pc`-th percentile of the training errors, and seismic association of the anomalies.

## Setup (PC or Google Colab)

- **PC:** set `LOCAL_PROJECT` to the repository folder.
- **Colab:** set `DRIVE_PROJECT` to the repository folder in your Google Drive. If it does not exist yet,
  it is cloned there from GitHub (code + data). An uploaded copy of the folder works too.

In [6]:
import sys
import subprocess
from pathlib import Path

LOCAL_PROJECT = Path(r"D:\GIT\Attention_Model-Demeter-Seismic-Study")
DRIVE_PROJECT = Path("/content/drive/MyDrive/[FuSe][STSM] Megha Babu/Attention model/Attention_Model-Demeter-Seismic-Study")
REPO_URL = "https://github.com/Meghababu99/Attention_Model-Demeter-Seismic-Study.git"
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = DRIVE_PROJECT
    if not PROJECT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT)], check=True)
else:
    PROJECT = LOCAL_PROJECT

CODE_DIR = PROJECT / "Clean-Code"
sys.path.insert(0, str(CODE_DIR))
print("Project:", PROJECT)

Project: D:\GIT\Attention_Model-Demeter-Seismic-Study


In [7]:
import random

import numpy as np
import pandas as pd
import torch

from lstm import (
    LSTMAutoencoder,
    SeismicCriteria,
    generate_windows,
    load_eq_catalog,
    prepare_window,
    train_window,
    evaluate_window,
)

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
from lstm.STSM import RecurrentAutoencoder, model_name

Device: cpu


## Models to train

In [8]:
MODELS = [
    # (CELL, BIDIRECTIONAL, ATTENTION)
    ("LSTM", False, "temporal"),          # LSTM-TA
    # ("LSTM", True,  "temporal"),          # BiLSTM-TA
    # ("GRU",  False, "temporal"),          # GRU-TA
    # ("GRU",  True,  "temporal"),          # BiGRU-TA
    # ("LSTM", False, "feature_temporal"),  # LSTM-FTA
    # ("LSTM", True,  "feature_temporal"),  # BiLSTM-FTA
    # ("GRU",  False, "feature_temporal"),  # GRU-FTA
    # ("GRU",  True,  "feature_temporal"),  # BiGRU-FTA
]
print("Models:", [model_name(*cfg) for cfg in MODELS])

Models: ['LSTM-TA']


## Settings (as the baseline training, except 20 windows and the cosine learning-rate schedule)

In [9]:
SEED = 201894
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Data
DATA_DIR = PROJECT / "Data"
BG_WINDOW_DIR = DATA_DIR / "Bg_window_data"
LABEL_DIR = DATA_DIR / "Label_data"
EQ_PATH = DATA_DIR / "EQ.csv"

# Rolling windows
train_months, val_months, stride = 12, 3, 3
WINDOW_START, WINDOW_END = "2005-01-01 00:00:00", "2011-01-02 00:00:00"
min_data_points = 17

# Anomaly threshold (percentile) and seismic criterion
pc = 98   # percentile
tw = 48   # time window (hours)
sw = 22   # spatial width (degrees)


# Model and training
input_size, hidden_size, num_layers, latent_dim = 11, 12, 2, 2
batch_size, lr, num_epochs, patience = 18, 0.0001, 1500, 100
RESTORE_BEST_EPOCH = True  # False: keep the model of the epoch where training stopped
# COSINE_T_MAX = 50           # CosineAnnealingLR(T_max=50), stepped every epoch
MATCH_A_RNG = True   # reproduce notebook A's random stream (extra evaluation pass per split)
WARM_START = True           # window i > 0 starts from the model of window i-1 (False = "In-" re-initialisation)
RESUME = False               # skip windows already in the result CSV (continues an interrupted run)

RUN = "A-SF" # SF means shuffle false

## Earthquake catalogue and rolling windows

In [10]:
eq = load_eq_catalog(EQ_PATH)
seismic_criteria = SeismicCriteria(spatial_width=sw, time_window_hours=tw)
windows_train = generate_windows(WINDOW_START, WINDOW_END, train_months, val_months, stride)

print("Earthquake catalogue shape:", eq.shape)
print("Number of rolling windows:", len(windows_train))
for i, w in enumerate(windows_train):
    print(f"Window {i}: Train {w['train_start']} -> {w['train_end']} | Val {w['val_start']} -> {w['val_end']}")

Earthquake catalogue shape: (12442, 5)
Number of rolling windows: 20
Window 0: Train 2005-01-01 00:00:00 -> 2006-01-01 00:00:00 | Val 2006-01-01 00:00:00 -> 2006-04-01 00:00:00
Window 1: Train 2005-04-01 00:00:00 -> 2006-04-01 00:00:00 | Val 2006-04-01 00:00:00 -> 2006-07-01 00:00:00
Window 2: Train 2005-07-01 00:00:00 -> 2006-07-01 00:00:00 | Val 2006-07-01 00:00:00 -> 2006-10-01 00:00:00
Window 3: Train 2005-10-01 00:00:00 -> 2006-10-01 00:00:00 | Val 2006-10-01 00:00:00 -> 2007-01-01 00:00:00
Window 4: Train 2006-01-01 00:00:00 -> 2007-01-01 00:00:00 | Val 2007-01-01 00:00:00 -> 2007-04-01 00:00:00
Window 5: Train 2006-04-01 00:00:00 -> 2007-04-01 00:00:00 | Val 2007-04-01 00:00:00 -> 2007-07-01 00:00:00
Window 6: Train 2006-07-01 00:00:00 -> 2007-07-01 00:00:00 | Val 2007-07-01 00:00:00 -> 2007-10-01 00:00:00
Window 7: Train 2006-10-01 00:00:00 -> 2007-10-01 00:00:00 | Val 2007-10-01 00:00:00 -> 2008-01-01 00:00:00
Window 8: Train 2007-01-01 00:00:00 -> 2008-01-01 00:00:00 | Val 20

## Training and evaluation of each model

In [11]:
def run_model(cell, bidirectional, attention):
    """Rolling-window training and evaluation of one model. Returns its result table."""
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    name = model_name(cell, bidirectional, attention)
    m = f"tw{tw}-30dBG_{'' if WARM_START else 'In-'}{RUN}-{name}"  # run label, used in every file name
    OUTPUT_DIR_RESULTS =  PROJECT/"Result Files" / "STSM-Results" / name / m / "outputs"
    OUTPUT_MODEL_DIR = PROJECT/"Result Files" / "STSM-Results" / name / m / "Model-Retraining"
    OUTPUT_DIR_RESULTS.mkdir(parents=True, exist_ok=True)
    OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    results_csv = OUTPUT_DIR_RESULTS / f"result_Hp_{m}.csv"
    print(f"\n{'#' * 90}\nModel: {name} | Run label: {m}\n{'#' * 90}")

    def build_model():
        return RecurrentAutoencoder(input_size, hidden_size, num_layers, latent_dim, cell, bidirectional, attention)

    done = pd.read_csv(results_csv) if RESUME and results_csv.exists() else pd.DataFrame(columns=["window"])
    all_results = done.to_dict("records")

    for i, w in enumerate(windows_train):
        if i in set(done["window"]):
            continue
        tag = f"Hp_{m}_w{i}"
        print(f"\n{'=' * 90}\nWindow {i}: Train {w['train_start']} -> {w['train_end']} | "
              f"Val {w['val_start']} -> {w['val_end']}\n{'=' * 90}")

        splits, datasets, scaled = prepare_window(
            BG_WINDOW_DIR / f"Background_data-window_{i}.pkl",
            {s: LABEL_DIR / f"summary_df_{s.lower()}_30D-{sw}SW-tw{tw}_w{i}.csv" for s in ("Train", "Val")},
            w,
            min_data_points,
        )

        prev_model = OUTPUT_MODEL_DIR / f"best_model_Hp_{m}_w{i-1}.pth" if WARM_START and i > 0 else None
        model = train_window(
            build_model(), scaled, tag, OUTPUT_MODEL_DIR, batch_size, prev_model_path = prev_model,
            num_epochs=num_epochs, lr=lr, patience=patience,
            restore_best_epoch=RESTORE_BEST_EPOCH, cosine_t_max=None ,save_all_epochs=False,
        )

        all_results += evaluate_window(i, w, model, splits, datasets, scaled, eq, seismic_criteria, pc, m, OUTPUT_DIR_RESULTS, match_a_rng=MATCH_A_RNG)
        pd.DataFrame(all_results).to_csv(results_csv, index=False)  # saved after every window

    return pd.DataFrame(all_results)

In [12]:
all_model_results = {model_name(*cfg): run_model(*cfg) for cfg in MODELS}


##########################################################################################
Model: LSTM-TA | Run label: tw48-30dBG_A-SF-LSTM-TA
##########################################################################################

Window 0: Train 2005-01-01 00:00:00 -> 2006-01-01 00:00:00 | Val 2006-01-01 00:00:00 -> 2006-04-01 00:00:00
Pairs: Train_normal=742, Val_normal=215, Train=1553, Val=381
Epoch [1/1500], Train Loss: 1.0118, Val Loss: 1.1952
Epoch [2/1500], Train Loss: 1.0104, Val Loss: 1.1932
Epoch [3/1500], Train Loss: 1.0093, Val Loss: 1.1914
Epoch [4/1500], Train Loss: 1.0082, Val Loss: 1.1898
Epoch [5/1500], Train Loss: 1.0073, Val Loss: 1.1884
Epoch [6/1500], Train Loss: 1.0064, Val Loss: 1.1870
Epoch [7/1500], Train Loss: 1.0055, Val Loss: 1.1857
Epoch [8/1500], Train Loss: 1.0047, Val Loss: 1.1844
Epoch [9/1500], Train Loss: 1.0039, Val Loss: 1.1832
Epoch [10/1500], Train Loss: 1.0031, Val Loss: 1.1818
Epoch [11/1500], Train Loss: 1.0019, Val Loss: 1.1801
Epoch [12/

KeyboardInterrupt: 

In [ ]:
all_model_results

NameError: name 'all_model_results' is not defined

## Results (mean over windows)

In [ ]:
summary = (
    pd.concat(all_model_results, names=["model"])
    .groupby(["model", "split"])[["anomalies", "agg_value", "agg_total_eq"]]
    .mean()
    .round(2)
)
summary.to_csv(CODE_DIR / "STSM-Results" / f"summary_tw{tw}-30dBG_{RUN}.csv")
display(summary)

OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive/[FuSe][STSM] Megha Babu/Attention model/Attention_Model-Demeter-Seismic-Study/Clean-Code/STSM-Results'